In [2]:
import numpy as np
from collections import defaultdict

class SeedAndExtend: # класс для реализации алгоритма Seed-and-Extend

    def __init__(self, k=4, X=2, match_score=3, mismatch_score=-3, gap_penalty=-2):

        self.k = k # длина seed
        self.X = X # порог X-drop
        self.match_score = match_score # балл за совпадение
        self.mismatch_score = mismatch_score # балл за несовпадение
        self.gap_penalty = gap_penalty # щтраф за гэп
        
    def build_index(self, database):
        
        index = defaultdict(list) # словарь {k-мер: [позиции вхождений]}
        n = len(database) # длина строки-референса (базы данных)
        
        for i in range(n - self.k + 1):
            kmer = database[i:i + self.k]
            index[kmer].append(i)
        
        return index
    
    def find_seeds(self, query, index):
        
        seeds = [] # список найденных семян (позиция в запросе, позиция в базе)
        m = len(query) # длина строки-запроса для поиска
        
        for i in range(m - self.k + 1):
            kmer = query[i:i + self.k]
            
            # ищем точные совпадения в индексе
            if kmer in index:
                for db_pos in index[kmer]:
                    seeds.append((i, db_pos))
        
        return seeds
    
    def extend_left(self, query, database, seed_q, seed_d, initial_score):
        
        left_aligned_q = []
        left_aligned_d = []
        
        # текущие позиции, начинаем с левого края seed
        i = seed_q - 1
        j = seed_d - 1
        
        # текущий счет и максимальный счет
        current_score = initial_score
        max_score = initial_score
        max_pos = (seed_q, seed_d)  # позиция максимального счета
        
        # матрица для локального выравнивания при расширении
        scores_history = [(0, 0, initial_score, "начало seed")]
        step = 0
        
        # продолжаем, пока не вышли за границы или не сработал X-drop
        while i >= 0 and j >= 0:
            step += 1
            # диагональ (совпадение/несовпадение)
            if query[i] == database[j]:
                diag = current_score + self.match_score
                match_type = "совпадение"
            else:
                diag = current_score + self.mismatch_score
                match_type = "несовпадение"
            
            new_score = diag
            
            drop = max_score - new_score
            
            left_aligned_q.append(query[i])
            left_aligned_d.append(database[j])
            
            # обновляем текущий счет
            current_score = new_score
            scores_history.append((i, j, current_score, match_type))
            if current_score > max_score:
                max_score = current_score
                max_pos = (i, j)
            
            # проверяем условие остановки X-drop
            if max_score - current_score >= self.X:
                break
            
            # двигаемся дальше влево
            i -= 1
            j -= 1
        
        # обрезаем выравнивание до позиции максимального счета
        if max_pos[0] <= seed_q - 1:  # если максимум достигнут где-то слева
            steps_from_seed = seed_q - max_pos[0]
            keep_chars = steps_from_seed
            left_aligned_q = left_aligned_q[-keep_chars:] if keep_chars > 0 else []
            left_aligned_d = left_aligned_d[-keep_chars:] if keep_chars > 0 else []
        
        return left_aligned_q, left_aligned_d, max_score, max_pos, scores_history
    
    def extend_right(self, query, database, seed_q, seed_d, initial_score):
        
        right_aligned_q = []
        right_aligned_d = []
        
        # текущие позиции, начинаем с правого края seed
        i = seed_q + self.k
        j = seed_d + self.k
        
        m = len(query)
        n = len(database)
        
        current_score = initial_score
        max_score = initial_score
        max_pos = (seed_q + self.k - 1, seed_d + self.k - 1)  # позиция максимального счета
        
        scores_history = [(seed_q + self.k - 1, seed_d + self.k - 1, initial_score, "начало seed")]
        step = 0
        
        while i < m and j < n:
            step += 1
            
            if query[i] == database[j]:
                diag = current_score + self.match_score
                match_type = "совпадение"
            else:
                diag = current_score + self.mismatch_score
                match_type = "несовпадение"
            
            new_score = diag
            drop = max_score - new_score
            
            right_aligned_q.append(query[i])
            right_aligned_d.append(database[j])
            
            current_score = new_score
            scores_history.append((i, j, current_score, match_type))
            
            if current_score > max_score:
                max_score = current_score
                max_pos = (i, j)
            
            if max_score - current_score >= self.X:
                break
            
            i += 1
            j += 1
        
        # обрезаем выравнивание до позиции максимального счета
        if max_pos[0] >= seed_q + self.k: 
            steps_from_seed = max_pos[0] - (seed_q + self.k - 1)
            keep_chars = steps_from_seed
            right_aligned_q = right_aligned_q[:keep_chars] if keep_chars > 0 else []
            right_aligned_d = right_aligned_d[:keep_chars] if keep_chars > 0 else []
        
        return right_aligned_q, right_aligned_d, max_score, max_pos, scores_history
    
    def extend_seed(self, query, database, seed_q, seed_d):
        
        # начальный счет - вес seed 
        initial_score = self.k * self.match_score
        
        left_q, left_d, left_max, left_max_pos, left_history = self.extend_left(query, database, seed_q, seed_d, initial_score)
        
        right_q, right_d, right_max, right_max_pos, right_history = self.extend_right(query, database, seed_q, seed_d, initial_score)
        
        # полное выравнивание
        full_q = list(reversed(left_q)) + [query[seed_q:seed_q+self.k]] + right_q
        full_d = list(reversed(left_d)) + [database[seed_d:seed_d+self.k]] + right_d
        full_q_str = ''.join(full_q)
        full_d_str = ''.join(full_d)
        
        # итоговый максимальный счет 
        final_max = max(left_max, right_max, initial_score)
        
        result = {
            'seed_q': seed_q,
            'seed_d': seed_d,
            'seed_seq': query[seed_q:seed_q+self.k],
            'initial_score': initial_score,
            'left_alignment': (''.join(reversed(left_q)), ''.join(reversed(left_d))),
            'right_alignment': (''.join(right_q), ''.join(right_d)),
            'full_alignment': (full_q_str, full_d_str),
            'left_max_score': left_max,
            'right_max_score': right_max,
            'final_max_score': final_max,
            'left_history': left_history,
            'right_history': right_history
        }
        
        return result
    
    def run(self, query, database):

        print(f"Параметры: k={self.k}, X={self.X}")
        print(f"Match={self.match_score}, Mismatch={self.mismatch_score}, Gap={self.gap_penalty}")
        print(f"\nРеференс (D): {database}")
        print(f"Запрос (Q): {query}")
        
        index = self.build_index(database)
        
        print(f"Индекс базы данных (k-меры длиной {self.k}):")
        for kmer, positions in sorted(index.items()):
            print(f"  '{kmer}': {positions}")
        
        seeds = self.find_seeds(query, index)
        
        if not seeds:
            print("Совпадений не найдено!")
            return None, None, index
        
        print(f"\nНайдено {len(seeds)} seed:")
        for i, (q_pos, d_pos) in enumerate(seeds):
            print(f"  Seed {i+1}: Q[{q_pos}] = '{query[q_pos:q_pos+self.k]}' -> D[{d_pos}]")
        
        all_results = []
        best_result = None
        best_score = -float('inf')
        
        for i, (q_pos, d_pos) in enumerate(seeds):
            print(f"\n--- Seed {i+1}/{len(seeds)} ---")
            result = self.extend_seed(query, database, q_pos, d_pos)
            all_results.append(result)
            
            print(f"\n  Результат для seed {i+1}:")
            print(f"    Левое расширение: max score = {result['left_max_score']}")
            print(f"    Правое расширение: max score = {result['right_max_score']}")
            print(f"    Итоговый max score = {result['final_max_score']}")
            
            if result['final_max_score'] > best_score:
                best_score = result['final_max_score']
                best_result = result
        
        return best_result, all_results, index
    
    def print_alignment(self, result):

        q_aligned, d_aligned = result['full_alignment']
        print("ИТОГОВОЕ ВЫРАВНИВАНИЕ")
        print(f"Seed: '{result['seed_seq']}' на позициях Q[{result['seed_q']}] и D[{result['seed_d']}]")
        print(f"Начальный счет seed: {result['initial_score']}")
        print(f"Максимальный счет: {result['final_max_score']}")
        
        print("\nВыравнивание:")
        print(f"Q: {q_aligned}")
        print(f"D: {d_aligned}")
        
        # строка совпадений
        match_str = ""
        matches = 0
        for a, b in zip(q_aligned, d_aligned):
            if a == b:
                match_str += "|"
                matches += 1
            else:
                match_str += " "
        
        print(f"   {match_str}")
        print(f"Совпадений: {matches}/{len(q_aligned)} ({matches/len(q_aligned)*100:.1f}%)")
        
        # позиции в оригинальных последовательностях
        q_start = result['seed_q'] - len(result['left_alignment'][0])
        d_start = result['seed_d'] - len(result['left_alignment'][1])
        
        print(f"\nПозиции в Q: {q_start}..{q_start + len(q_aligned) - 1}")
        print(f"Позиции в D: {d_start}..{d_start + len(d_aligned) - 1}")



if __name__ == "__main__":
    
    # входные данные и параметрв
    database = "CTAGGATCCAGGCATACGA"
    query = "GGATCCATTCATTA"
    k = 4
    X = 2
    
    aligner = SeedAndExtend(k=k, X=X)
    best_result, all_results, index = aligner.run(query, database)
    
    if best_result:

        aligner.print_alignment(best_result)
        
        print(f"\nЛучший seed: '{best_result['seed_seq']}'")
        print(f"Позиция в Q: {best_result['seed_q']}")
        print(f"Позиция в D: {best_result['seed_d']}")
        
        print("\nРасширения влево:")
        print("  Шаг | Позиция | Счет | Событие")
        print("-" * 40)
        for i, (q_pos, d_pos, score, event) in enumerate(best_result['left_history']):
            if i == 0:
                print(f"  {i:3} | ({q_pos},{d_pos:2}) | {score:4} | {event}")
            else:
                print(f"  {i:3} | ({q_pos},{d_pos:2}) | {score:4} | {event}")
        print(f"  Максимальный счет слева: {best_result['left_max_score']}")
        
        print("\nРасширения вправо:")
        print("  Шаг | Позиция | Счет | Событие")
        print("-" * 40)
        for i, (q_pos, d_pos, score, event) in enumerate(best_result['right_history']):
            if i == 0:
                print(f"  {i:3} | ({q_pos},{d_pos:2}) | {score:4} | {event}")
            else:
                print(f"  {i:3} | ({q_pos},{d_pos:2}) | {score:4} | {event}")
        print(f"  Максимальный счет справа: {best_result['right_max_score']}")
        print("\nЛевое расширение:")
        left_q, left_d = best_result['left_alignment']
        print(f"  Q: {left_q}")
        print(f"  D: {left_d}")
        
        print("\nПравое расширение:")
        right_q, right_d = best_result['right_alignment']
        print(f"  Q: {right_q}")
        print(f"  D: {right_d}")
    else:
        print("\nНе найдено ни одного подходящего выравнивания.")

Параметры: k=4, X=2
Match=3, Mismatch=-3, Gap=-2

Референс (D): CTAGGATCCAGGCATACGA
Запрос (Q): GGATCCATTCATTA
Индекс базы данных (k-меры длиной 4):
  'ACGA': [15]
  'AGGA': [2]
  'AGGC': [9]
  'ATAC': [13]
  'ATCC': [5]
  'CAGG': [8]
  'CATA': [12]
  'CCAG': [7]
  'CTAG': [0]
  'GATC': [4]
  'GCAT': [11]
  'GGAT': [3]
  'GGCA': [10]
  'TACG': [14]
  'TAGG': [1]
  'TCCA': [6]

Найдено 4 seed:
  Seed 1: Q[0] = 'GGAT' -> D[3]
  Seed 2: Q[1] = 'GATC' -> D[4]
  Seed 3: Q[2] = 'ATCC' -> D[5]
  Seed 4: Q[3] = 'TCCA' -> D[6]

--- Seed 1/4 ---

  Результат для seed 1:
    Левое расширение: max score = 12
    Правое расширение: max score = 21
    Итоговый max score = 21

--- Seed 2/4 ---

  Результат для seed 2:
    Левое расширение: max score = 15
    Правое расширение: max score = 18
    Итоговый max score = 18

--- Seed 3/4 ---

  Результат для seed 3:
    Левое расширение: max score = 18
    Правое расширение: max score = 15
    Итоговый max score = 18

--- Seed 4/4 ---

  Результат для see